# Notebook 06c — Attack Validation and Analysis

After running the attack in NB06b and getting a very high success rate, I wanted to evaluate the quality of the attack, by evaluating whether someone could realistically detect what is happening.

By looking at how other people validate adversarial attacks on sensor signals, I ended up testing four different things. Each one came from a different question I had after looking at the NB06b numbers:

1. **Spectral analysis**: the perturbations are small in L2 norm, but are they actually hiding in the natural frequency range of gait? Or would they show up as obvious high-frequency noise?
2. **K ablation**: I picked K=20 PGD iterations somewhat arbitrarily. Was that actually enough, or would more iterations have changed the ε_min estimates?
3. **Detection resistance**: could a simple detector based on signal frequency content catch these adversarial examples, even if the model cannot?
4. **Feature-space shift**: does the attack actually move the probe signal toward the target inside the CNN, or is it just tricking the final classifier layer through some shortcut?

In [ ]:
import sys
sys.path.insert(0, '..')

import logging
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import welch
from scipy.stats import mannwhitneyu

import torch

from src.data.auth_dataset  import load_auth_dataset, normalize_auth
from src.models.gait_cnn   import GaitCNN
from src.models.auth_model import AuthModel
from src.attacks.pgd import (
    pgd_sensor_variable_eps,
    run_pgd_sensor_batched,
    batch_psame,
)
from src.utils.latex_writer import write_latex_metrics

# ── Dataset configuration ─────────────────────────────────────────────────────
DATASET_NAME = 'Dataset #5'
DATA_ROOT    = Path('../data')
DATASET_PATH = DATA_ROOT / DATASET_NAME
NORM_PATH    = Path('../logs/auth_norm_stats.npz')
LOG_DIR      = Path('../logs')
CKPT_DIR     = Path('../checkpoints')
RESULT_DIR   = Path('../results')

# Sensor parameters
N_CHANNELS  = 6     # acc_x, acc_y, acc_z, gyr_x, gyr_y, gyr_z
N_TIMESTEPS = 128
FS          = 50.0  # Hz — sampling rate of the IMU

# Validation config
N_SAMPLE_SPECTRAL = 200   # pairs used for spectral + detection analysis
N_SAMPLE_ABLATION = 100   # pairs used for K ablation
K_VALUES          = [5, 10, 20, 40, 80]  # PGD iterations to sweep
SEED_SAMPLE       = 0
BATCH             = 256
DEVICE            = torch.device('cpu')

log = logging.getLogger('nb06c')
log.setLevel(logging.DEBUG)
log.handlers.clear()
fh = logging.FileHandler(LOG_DIR / f'06c_{DATASET_NAME.replace(" ","_")}_validation.log', mode='w')
fh.setFormatter(logging.Formatter('%(asctime)s  %(message)s', datefmt='%H:%M:%S'))
sh = logging.StreamHandler()
sh.setFormatter(logging.Formatter('%(message)s'))
log.addHandler(fh); log.addHandler(sh)
log.info(f'=== Notebook 06c — Attack Validation  [{DATASET_NAME}] ===')

## Section 1 — Load Attack Results and Data

In [ ]:
# ── NB06b test results ────────────────────────────────────────────────────────
res = np.load(LOG_DIR / f'06b_{DATASET_NAME.replace(" ","_")}_attack_results.npz',
              allow_pickle=False)
pair_succeeded   = res['pair_succeeded']
pair_eps_min     = res['pair_eps_min']
pair_actual_l2   = res['pair_actual_l2']
pair_psr         = res['pair_psr']
eps_target       = float(res['eps_target'])
K_PGD_ORIGINAL   = int(res['k_pgd'])
N_BISECT         = int(res['n_bisect'])
combo_l2         = res['combo_l2']

# ── Raw test split ────────────────────────────────────────────────────────────
norm_stats        = np.load(NORM_PATH)
norm_mean, norm_std = norm_stats['mean'], norm_stats['std']

X1_te, X2_te, y_te = load_auth_dataset(str(DATASET_PATH), 'test')
X1_te_n, X2_te_n, _ = normalize_auth(X1_te, X2_te, (norm_mean, norm_std))

diff_mask = y_te == 1
x1_diff   = X1_te_n[diff_mask]
x2_diff   = X2_te_n[diff_mask]
N_PAIRS   = len(x1_diff)

X1_diff_t = torch.from_numpy(x1_diff).float()
X2_diff_t = torch.from_numpy(x2_diff).float()

# ── Model ─────────────────────────────────────────────────────────────────────
cnn = GaitCNN(n_classes=98)
cnn.load_state_dict(torch.load(CKPT_DIR / 'cnn_encoder.pt', map_location='cpu'))
cnn.eval()
auth_model = AuthModel(cnn).to(DEVICE)
auth_model.load_state_dict(torch.load(CKPT_DIR / 'auth_model.pt', map_location='cpu'))
auth_model.eval()

# ── Select solved pairs for analysis ─────────────────────────────────────────
solved_idx = np.where(pair_succeeded)[0]
rng        = np.random.default_rng(SEED_SAMPLE)
sample_idx = rng.choice(solved_idx, size=min(N_SAMPLE_SPECTRAL, len(solved_idx)), replace=False)
sample_idx = np.sort(sample_idx)

log.info(f'Loaded: {N_PAIRS} test pairs  |  solved={pair_succeeded.sum()}  |'
         f'  analysis sample={len(sample_idx)}')
print(f'Test pairs: {N_PAIRS}  |  solved: {pair_succeeded.sum()}  |  '
      f'analysis sample: {len(sample_idx)}')

## Section 2 — Spectral Analysis

One of the first things I wanted to check was whether the perturbation looks like natural gait variation or artificial noise. The concern is that even if ‖δ‖₂ is small, the perturbation might be concentrated at high frequencies such as random jitter that a real sensor would never produce.

Gait signals have a very recognisable frequency structure: the dominant energy sits at around 1–2 Hz (roughly one full step cycle per second), with harmonics at 2–4 Hz. Above ~5 Hz there is essentially no gait-related signal; anything there is sensor noise. So I thought: if the perturbation PSD looks like the gait PSD, peaked in the low-frequency range, then it is hard to distinguish from natural intra-session variation. If it is flat or peaks at high frequencies, a simple frequency filter could in theory remove it.

The idea is to split the signal into overlapping segments, compute an FFT on each one, and average the squared magnitudes. This gives a much smoother estimate than a single FFT on the whole window, which tends to be noisy. I found this approach in scipy (`scipy.signal.welch`) and it is widely used in the biosignal processing literature.

For each of the 200 sampled pairs I reconstruct x1_adv by re-running PGD at the saved ε_min, then compare the PSD of x1_clean, x1_adv, and the perturbation δ = x1_adv − x1_clean across all six IMU channels.

In [3]:
# Reconstruct adversarial examples at the saved ε_min for the analysis sample
eps_sample = torch.tensor(pair_eps_min[sample_idx], dtype=torch.float32)
x1_sample  = X1_diff_t[sample_idx]
x2_sample  = X2_diff_t[sample_idx]

log.info(f'Re-running PGD at saved ε_min for {len(sample_idx)} pairs (K={K_PGD_ORIGINAL})')

with torch.no_grad():
    pass  # adversarial generation needs grad

# Run in mini-batches to avoid OOM
adv_parts = []
for s in range(0, len(sample_idx), BATCH):
    adv_parts.append(
        pgd_sensor_variable_eps(
            auth_model,
            x1_sample[s:s+BATCH], x2_sample[s:s+BATCH],
            target_label=0,
            eps_arr=eps_sample[s:s+BATCH],
            K=K_PGD_ORIGINAL,
        )
    )
x1_adv_sample = torch.cat(adv_parts, dim=0)

# Verify attack success on reconstructed examples
scores_check = batch_psame(auth_model, x1_adv_sample, x2_sample, batch_size=BATCH)
asr_check    = float((scores_check < 0.5).mean())
log.info(f'Re-run ASR check: {asr_check:.3f} (should match original ~1.0)')
print(f'ASR on reconstructed adversarial examples: {asr_check:.3f}')

x1_np  = x1_sample.numpy()          # (N, 6, 128) clean
x1a_np = x1_adv_sample.numpy()      # (N, 6, 128) adversarial
d_np   = x1a_np - x1_np             # (N, 6, 128) perturbation

Re-running PGD at saved ε_min for 200 pairs (K=20)
Re-run ASR check: 0.995 (should match original ~1.0)


ASR on reconstructed adversarial examples: 0.995


In [ ]:
# Compute mean PSD across pairs for each channel group
# Channels: 0-2 = accelerometer, 3-5 = gyroscope
NPERSEG = 64  # Welch segment length (half the window → 3 dB freq resolution)

freqs, _ = welch(x1_np[0, 0], fs=FS, nperseg=NPERSEG)
n_freq   = len(freqs)

psd_clean = np.zeros((N_SAMPLE_SPECTRAL, N_CHANNELS, n_freq))
psd_adv   = np.zeros((N_SAMPLE_SPECTRAL, N_CHANNELS, n_freq))
psd_delta = np.zeros((N_SAMPLE_SPECTRAL, N_CHANNELS, n_freq))

for i in range(len(sample_idx)):
    for ch in range(N_CHANNELS):
        _, p_c = welch(x1_np[i, ch],  fs=FS, nperseg=NPERSEG)
        _, p_a = welch(x1a_np[i, ch], fs=FS, nperseg=NPERSEG)
        _, p_d = welch(d_np[i, ch],   fs=FS, nperseg=NPERSEG)
        psd_clean[i, ch] = p_c
        psd_adv[i, ch]   = p_a
        psd_delta[i, ch] = p_d

# Average over pairs
mean_clean = psd_clean.mean(axis=0)   # (6, n_freq)
mean_adv   = psd_adv.mean(axis=0)
mean_delta = psd_delta.mean(axis=0)

ch_names = ['acc_x', 'acc_y', 'acc_z', 'gyr_x', 'gyr_y', 'gyr_z']
colors   = {'clean': '#3498db', 'adv': '#e74c3c', 'delta': '#2ecc71'}

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=False)
axes_flat = axes.flatten()

for ch, ax in enumerate(axes_flat):
    ax.semilogy(freqs, mean_clean[ch], color=colors['clean'],
                linewidth=1.8, label='clean x1')
    ax.semilogy(freqs, mean_adv[ch],   color=colors['adv'],
                linewidth=1.8, linestyle='--', label='adversarial x1')
    ax.semilogy(freqs, mean_delta[ch], color=colors['delta'],
                linewidth=1.5, linestyle=':', label='δ (perturbation)')
    ax.set_title(ch_names[ch])
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PSD')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle(
    f'Power Spectral Density — clean vs adversarial [{DATASET_NAME}]\n'
    f'mean over {len(sample_idx)} successfully attacked pairs  |  K={K_PGD_ORIGINAL}',
    fontsize=11,
)
plt.tight_layout()
plt.savefig(RESULT_DIR / f'06c_{DATASET_NAME.replace(" ","_")}_spectral.png', dpi=150)
plt.show()

# Fraction of perturbation energy in low-frequency band (0–5 Hz = gait band)
gait_band   = freqs <= 5.0
frac_in_band = float(
    mean_delta[:, gait_band].sum() / mean_delta.sum()
)
log.info(f'Perturbation energy in gait band (0–5 Hz): {frac_in_band:.3f} ({frac_in_band*100:.1f}%)')
print(f'Perturbation energy in gait band (0–5 Hz): {frac_in_band*100:.1f}%')

## Section 3 — K Ablation

When I set K=20 PGD iterations in NB06b, I did not verified if it was the right choice for this specific setting. There are two things that could go wrong:

- Too few iterations: the attack does not have enough steps to find the minimum budget, so ε_min is overestimated.
- Too many iterations: the attack is doing extra work for no gain, and the results from NB06b could have been obtained with fewer calls.

To check this, I ran the full grid sweep plus binary search on a fixed sample of 100 solved pairs, varying K across {5, 10, 20, 40, 80}, and looked at how mean ε_min and ASR change with K.

The idea is straightforward: if the results are still changing significantly as K increases, we need more iterations. If they plateau, we know K=20 was at or past the knee of the curve and the original results are reliable.

In [ ]:
# Select a stable sample of solved pairs for the ablation
rng_ab     = np.random.default_rng(SEED_SAMPLE + 1)
ablation_idx = rng_ab.choice(solved_idx, size=min(N_SAMPLE_ABLATION, len(solved_idx)),
                              replace=False)
ablation_idx = np.sort(ablation_idx)

x1_ab = X1_diff_t[ablation_idx]
x2_ab = X2_diff_t[ablation_idx]

# Load the ε grid from the NB06a setup file
setup     = np.load(LOG_DIR / f'06a_{DATASET_NAME.replace(" ","_")}_attack_setup.npz',
                    allow_pickle=False)
EPS_GRID  = setup['eps_grid'].astype(np.float64)

log.info(f'K ablation: {len(ablation_idx)} pairs × {len(K_VALUES)} K values')
print(f'Running K ablation: {len(ablation_idx)} pairs, K ∈ {K_VALUES}')

ablation_results = {}  # K → {'mean_eps_min': float, 'asr': float, 'eps_min_arr': np.array}

for K in K_VALUES:
    # Grid sweep
    succeeded = np.zeros(len(ablation_idx), dtype=bool)
    lo        = np.zeros(len(ablation_idx), dtype=np.float32)
    hi        = np.full(len(ablation_idx), np.inf, dtype=np.float32)
    prev_eps  = 0.0

    for eps in EPS_GRID:
        pending = np.where(~succeeded)[0]
        if len(pending) == 0:
            break
        x1_pend = x1_ab[pending]
        x2_pend = x2_ab[pending]
        x1_adv  = run_pgd_sensor_batched(
            auth_model, x1_pend, x2_pend, target_label=0,
            eps=float(eps), K=K, batch_size=BATCH,
        )
        scores  = batch_psame(auth_model, x1_adv, x2_pend, batch_size=BATCH)
        newly   = pending[scores < 0.5]
        succeeded[newly] = True
        lo[newly] = float(prev_eps)
        hi[newly] = float(eps)
        prev_eps  = float(eps)

    # Binary search
    brack  = np.where(succeeded)[0]
    lo_t   = torch.tensor(lo[brack], dtype=torch.float32)
    hi_t   = torch.tensor(hi[brack], dtype=torch.float32)
    x1_br  = x1_ab[brack]
    x2_br  = x2_ab[brack]

    for _ in range(N_BISECT):
        mid = (lo_t + hi_t) / 2
        x1_mid = pgd_sensor_variable_eps(
            auth_model, x1_br, x2_br, target_label=0, eps_arr=mid, K=K
        )
        s_mid  = torch.from_numpy(batch_psame(auth_model, x1_mid, x2_br, batch_size=BATCH))
        ok     = s_mid < 0.5
        hi_t   = torch.where(ok, mid, hi_t)
        lo_t   = torch.where(ok, lo_t, mid)

    eps_min_arr = np.full(len(ablation_idx), np.nan)
    eps_min_arr[brack] = hi_t.numpy()

    asr = float(succeeded.mean())
    mean_eps = float(np.nanmean(eps_min_arr))
    ablation_results[K] = {'mean_eps_min': mean_eps, 'asr': asr,
                            'eps_min_arr': eps_min_arr}
    log.info(f'K={K:3d}: ASR={asr:.3f}  mean_eps_min={mean_eps:.4f}')
    print(f'  K={K:3d}: ASR={asr:.3f}  mean ε_min={mean_eps:.4f}')

In [ ]:
k_vals    = sorted(ablation_results.keys())
mean_eps  = [ablation_results[k]['mean_eps_min'] for k in k_vals]
asr_vals  = [ablation_results[k]['asr'] for k in k_vals]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(k_vals, mean_eps, 'o-', color='#3498db', linewidth=2, markersize=7)
ax.axvline(K_PGD_ORIGINAL, color='#e74c3c', linewidth=1.5, linestyle='--',
           label=f'original K={K_PGD_ORIGINAL}')
ax.set_xlabel('K (PGD iterations)')
ax.set_ylabel('Mean ε_min')
ax.set_title('K ablation — minimum budget vs PGD depth\n(lower = attack needs less perturbation)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(k_vals, asr_vals, 's-', color='#2ecc71', linewidth=2, markersize=7)
ax.axvline(K_PGD_ORIGINAL, color='#e74c3c', linewidth=1.5, linestyle='--',
           label=f'original K={K_PGD_ORIGINAL}')
ax.set_ylim(0, 1.05)
ax.set_xlabel('K (PGD iterations)')
ax.set_ylabel('Attack Success Rate')
ax.set_title(f'K ablation — ASR vs PGD depth\n({len(ablation_idx)} pairs)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle(f'K ablation [{DATASET_NAME}]', fontsize=11)
plt.tight_layout()
plt.savefig(RESULT_DIR / f'06c_{DATASET_NAME.replace(" ","_")}_k_ablation.png', dpi=150)
plt.show()

# Convergence ratio: ε_min(K_max) / ε_min(K_min)
k_min_val, k_max_val = k_vals[0], k_vals[-1]
conv_ratio = ablation_results[k_max_val]['mean_eps_min'] / ablation_results[k_min_val]['mean_eps_min']
log.info(f'Convergence ratio ε_min({k_max_val}) / ε_min({k_min_val}) = {conv_ratio:.3f}')
print(f'Convergence ratio ε_min(K={k_max_val}) / ε_min(K={k_min_val}) = {conv_ratio:.3f}')

## Section 4 — Detection Resistance

Even if the authentication model is fooled, a system could still detect the attack by inspecting the signal before it reaches the model. The most obvious approach is a frequency-based detector: adversarial perturbations added naively on top of a natural signal often introduce high-frequency energy that is not present in the original recording. For an IMU gait signal, anything above ~5–10 Hz is already outside the biological range of human movement, so extra energy there would be a red flag.

I implemented the simplest possible version of this: the **high-frequency energy ratio** (HFR), defined as the fraction of total signal energy that falls above a cutoff of 10 Hz. A clean gait window should have a low HFR; an adversarial signal with high-frequency noise should have a higher one.

To test whether the two distributions are actually different, a p-value above 0.05 means the test cannot tell the two groups apart, which is exactly what we want for the attack to remain stealthy.

In [ ]:
F_CUTOFF = 10.0  # Hz — above gait band, below Nyquist

hf_mask = freqs > F_CUTOFF

# HFR per pair per channel, then average channels
hfr_clean = np.zeros(len(sample_idx))
hfr_adv   = np.zeros(len(sample_idx))

for i in range(len(sample_idx)):
    ch_hfr_c, ch_hfr_a = [], []
    for ch in range(N_CHANNELS):
        _, pc = welch(x1_np[i, ch],  fs=FS, nperseg=NPERSEG)
        _, pa = welch(x1a_np[i, ch], fs=FS, nperseg=NPERSEG)
        ch_hfr_c.append(pc[hf_mask].sum() / (pc.sum() + 1e-12))
        ch_hfr_a.append(pa[hf_mask].sum() / (pa.sum() + 1e-12))
    hfr_clean[i] = np.mean(ch_hfr_c)
    hfr_adv[i]   = np.mean(ch_hfr_a)

stat, p_val = mannwhitneyu(hfr_clean, hfr_adv, alternative='two-sided')
distinguishable = p_val < 0.05

fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(0, max(hfr_clean.max(), hfr_adv.max()) * 1.05, 50)
ax.hist(hfr_clean, bins=bins, alpha=0.6, color='#3498db',
        label=f'clean  μ={hfr_clean.mean():.4f}')
ax.hist(hfr_adv,   bins=bins, alpha=0.6, color='#e74c3c',
        label=f'adversarial  μ={hfr_adv.mean():.4f}')
ax.set_xlabel(f'High-frequency energy ratio (f > {F_CUTOFF} Hz)')
ax.set_ylabel('Pair count')
ax.set_title(
    f'Detection resistance [{DATASET_NAME}]\n'
    f'Mann–Whitney U: p={p_val:.4f}  →  '
    + ('DISTINGUISHABLE (detector works)' if distinguishable
       else 'NOT distinguishable (detector fails)')
)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULT_DIR / f'06c_{DATASET_NAME.replace(" ","_")}_detection.png', dpi=150)
plt.show()

log.info(f'HFR: clean μ={hfr_clean.mean():.4f}  adv μ={hfr_adv.mean():.4f}  '
         f'Mann-Whitney p={p_val:.4f}  distinguishable={distinguishable}')
print(f'HFR clean={hfr_clean.mean():.4f}  adv={hfr_adv.mean():.4f}  '
      f'p={p_val:.4f}  detector: {"works" if distinguishable else "fails"}')

## Section 5 — Feature-Space Shift

The last thing I wanted to understand is what the attack is actually doing inside the model. The authentication model works in two stages: a CNN encoder that compresses the raw signal into a feature representation, and an LSTM+FC classifier that operates on those features. The attack optimises the input signal end-to-end, but that does not tell us which stage is actually being exploited.

There are two very different things the attack could be doing:

- **Moving toward the target in feature space**: x1_adv's CNN representation actually becomes closer to x2_B's. The model is being fooled because the probe now looks like the target at the representation level.
- **Exploiting the decision boundary only**: the CNN representation barely changes, but the perturbation is enough to push the pair's score across the 0.5 boundary — a shortcut through the geometry of the last classifier layer.

To distinguish these, I extracted the CNN feature maps for x1_clean, x1_adv, and x2_B, and measured the **Frobenius norm** (square root of the sum of squared entries) of the difference between each probe and the reference. A natural way to measure distance between matrices (the feature maps here are 2D tensors of shape 16×128). I then computed the ratio d_adv/d_clean: a value below 1 means the adversarial example is closer to the target in feature space; a value above 1 means it is actually further away.

In [ ]:
# Extract CNN features (B, 16, 128) via the encoder only
def encode_batched(cnn_model, x: torch.Tensor, batch_size: int = 256) -> torch.Tensor:
    parts = []
    with torch.no_grad():
        for s in range(0, x.shape[0], batch_size):
            feat = cnn_model.encode(x[s:s+batch_size])
            parts.append(feat)
    return torch.cat(parts, dim=0)

f_clean = encode_batched(cnn, x1_sample)
f_adv   = encode_batched(cnn, x1_adv_sample)
f_ref   = encode_batched(cnn, x2_sample)

# Frobenius distance in feature space
d_clean = (f_clean - f_ref).flatten(1).norm(dim=1).numpy()
d_adv   = (f_adv   - f_ref).flatten(1).norm(dim=1).numpy()

ratio = d_adv / (d_clean + 1e-8)
closer_frac = float((d_adv < d_clean).mean())
mean_ratio  = float(ratio.mean())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.scatter(d_clean, d_adv, s=8, alpha=0.4, color='#3498db')
lim = max(d_clean.max(), d_adv.max()) * 1.05
ax.plot([0, lim], [0, lim], 'k--', linewidth=1, label='d_adv = d_clean (no change)')
ax.set_xlabel('‖CNN(x1) − CNN(x2_B)‖  (clean)')
ax.set_ylabel('‖CNN(x1_adv) − CNN(x2_B)‖  (adversarial)')
ax.set_title(f'Feature-space shift\n'
             f'Points below diagonal: moved closer to target ({closer_frac*100:.1f}% of pairs)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.hist(ratio, bins=60, color='#2ecc71', alpha=0.8, edgecolor='none')
ax.axvline(1.0, color='black', linewidth=1.5, linestyle='--', label='ratio = 1 (no change)')
ax.axvline(mean_ratio, color='#e74c3c', linewidth=1.5,
           label=f'mean ratio = {mean_ratio:.3f}')
ax.set_xlabel('d_adv / d_clean (ratio < 1 = moved toward target)')
ax.set_ylabel('Pair count')
ax.set_title('Feature-space distance ratio\n(< 1 means attack found the right direction)')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle(f'Feature-space shift [{DATASET_NAME}]', fontsize=11)
plt.tight_layout()
plt.savefig(RESULT_DIR / f'06c_{DATASET_NAME.replace(" ","_")}_feature_shift.png', dpi=150)
plt.show()

log.info(f'Feature shift: closer_frac={closer_frac:.3f}  mean_ratio={mean_ratio:.3f}')
print(f'Closer in feature space: {closer_frac*100:.1f}% of pairs\n'
      f'Mean d_adv / d_clean ratio: {mean_ratio:.3f}')

## Section 6 — Save Validation Metrics

In [ ]:
tag = DATASET_NAME.replace(' ', '_').replace('#', '')

k_best_eps = min(ablation_results, key=lambda k: ablation_results[k]['mean_eps_min'])

write_latex_metrics(f'nb06c_{tag}', {
    # Spectral
    'spectralFracInBand':    f'{frac_in_band*100:.1f}',
    'spectralFCutoff':       f'{F_CUTOFF:.0f}',
    # Detection
    'detectionPValue':       f'{p_val:.4f}',
    'detectionDistinguish':  'yes' if distinguishable else 'no',
    'hfrCleanMean':          f'{hfr_clean.mean():.4f}',
    'hfrAdvMean':            f'{hfr_adv.mean():.4f}',
    # K ablation
    'kValues':               str(K_VALUES),
    'kOriginal':             str(K_PGD_ORIGINAL),
    'kConvergenceRatio':     f'{conv_ratio:.3f}',
    'kBestEps':              str(k_best_eps),
    'kBestMeanEpsMin':       f'{ablation_results[k_best_eps]["mean_eps_min"]:.4f}',
    # Feature shift
    'featureCloserFrac':     f'{closer_frac*100:.1f}',
    'featureMeanRatio':      f'{mean_ratio:.3f}',
    # Sample sizes
    'nSampleSpectral':       str(len(sample_idx)),
    'nSampleAblation':       str(len(ablation_idx)),
}, output_dir='../latex/generated', log=log)

print('LaTeX metrics written.')
log.info('=== NB06c complete ===')

## Section 7 — Results Summary

### Test split (3,800 pairs, 220 subject-pair combos)

The PGD attack achieved a pair-level attack success rate of 99.97% (3,799/3,800 pairs).
Of the 3,800 pairs, 982 (25.84%) were accepted by the authentication model at ε = 0
(baseline impostor acceptance), so the attack was required for the remaining 74.2%.

The mean minimum L2 budget ε_min was 2.90 (11.8% of ε_target = 24.54); the median was
0.82 (3.4%).  The 25th percentile was 0.019 and the 90th percentile was 6.53, indicating
a heavily right-skewed distribution where most pairs are fooled with very small
perturbations.  The mean perturbation-to-signal ratio (PSR = ‖δ‖₂/‖x1‖₂) was 7.13%;
the median PSR was 2.92%.

### Difficulty quartile breakdown

Splitting the 220 combos by inter-subject gait distance (L2) into four equal quartiles
gives a monotonically increasing median ε_min: Q1 = 0.30, Q2 = 0.86, Q3 = 0.82,
Q4 = 1.43.  The slight non-monotonicity at Q2/Q3 is within sampling noise; the
overall trend confirms that structurally similar subject pairs require smaller budgets.

### Training split (500 of 4,753 combos sampled, seed = 42)

On the enrolled-member (training) split the attack reached ASR = 100% across all 500
sampled combos.  Mean ε_min was 4.69 (19.1% of ε_target), mean PSR was 12.53%; roughly
1.6× larger than on the test split.  The higher cost on training subjects is consistent
with the model having better-calibrated boundaries for subjects it was trained on.

### Spectral and detection properties

68.9% of the perturbation energy fell in the gait frequency band (0–5 Hz).
A Mann–Whitney U test on the high-frequency energy ratio (f > 10 Hz) between clean and
adversarial signals gave p = 0.607, indicating the two distributions are not
statistically distinguishable and a naive spectral detector cannot flag the attack.

### K ablation

Across K ∈ {5, 10, 20, 40, 80} PGD iterations on 100 sampled pairs:
K = 5 achieved ASR = 99% with mean ε_min = 6.48; K = 10 reached 100% at 4.34;
K = 20 reached 100% at 2.55; K = 40 at 2.31; K = 80 at 2.21.
The convergence ratio ε_min(K=80) / ε_min(K=5) = 0.34, and the curve is near-flat
from K = 20 onward, identifying K = 20 as the knee point used throughout.

### Feature-space shift

58.5% of the 200 sampled pairs moved closer to the target in CNN feature space after
perturbation.  The mean ratio d_adv / d_clean = 1.00, indicating that the attack
exploits the classifier's decision boundary without substantially closing the
representation-space gap between the two subjects.